# Part E: Main model and evaluation

**Question.** Which product and reviewer characteristics are associated with the likelihood of a five-star review, and does the price association differ by skin type?

**Two separate jobs in this notebook**

| Job | Data used | Answers |
| --- | --- | --- |
| Inference (Sections 3–6) | All reviews | Which factors are associated with five-star ratings? Odds ratios, clustered 95% CIs, interaction test |
| Evaluation (Section 7) | 80/20 split **by product** | How much better than a naive baseline does the model rank reviews? |

Inference uses all the data because the goal is to estimate associations as precisely as possible. Evaluation needs held-out products, so the model is never scored on products it was fitted on.

Input: `data/processed/analysis_ready.parquet` from Part B. Variable definitions are in `feature_dictionary.csv`.
**Evaluation scope:** the processed features were prepared globally, including author attribute fills across products. Product-disjoint fitting is verified; strict prospective preprocessing isolation is not established (see docs/REVIEW.md).


### 1. Setup

In [ ]:
# Find only this project (never search the user's home directory).
import sys
from pathlib import Path
_PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_paths.py").is_file()), None)
if _PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook within the project folder.")
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
from project_paths import ROOT, RESULTS_DIR, ANALYSIS_PATH, RAW_DIR, PROCESSED_DIR, require_analysis, require_merged

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (roc_auc_score, f1_score, balanced_accuracy_score,
                             confusion_matrix, accuracy_score)

SEED = 5106
BASE_DIR = ROOT
DATA_PATH = require_analysis()
OUT_DIR = RESULTS_DIR / "part_e"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Plot style: one validated categorical palette, recessive axes
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "font.size": 10,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
})

### 2. Load data and prepare model variables

In [ ]:
df = pd.read_parquet(DATA_PATH)
print("Rows:", f"{len(df):,}", "| Products:", df["product_id"].nunique(), "| Five-star rate:", round(df["five_star"].mean(), 4))

# Short labels so model terms are readable in tables and plots
PRICE_MAP = {"Budget (<$25)": "Budget", "Mid ($25-49)": "Mid", "Premium ($50-99)": "Premium", "Luxury ($100+)": "Luxury"}
PRICE_ORDER = ["Budget", "Mid", "Premium", "Luxury"]
SKIN_ORDER = ["Normal", "Combination", "Dry", "Oily", "Not reported"]

df["price"] = df["price_segment"].astype(str).map(PRICE_MAP)
df["skin"] = df["skin_type_cat"].astype(str)
df["cat"] = df["category"].astype(str)
assert df["price"].notna().all() and set(df["skin"]) == set(SKIN_ORDER)

HL = ["hl_fragrance_free", "hl_clean", "hl_vegan", "hl_cruelty_free",
      "hl_hydrating", "hl_dullness", "hl_dryness", "hl_anti_aging", "highlights_missing"]

# Leakage guard: none of these may appear in the model
LEAKAGE = ["rating", "recommended", "review_text", "review_title", "helpfulness",
           "loves_count", "product_rating", "product_reviews"]
MAIN_TERMS = ["price", "skin", "cat"] + HL
assert not set(LEAKAGE) & set(MAIN_TERMS)

# Product IDs as integer codes for clustered standard errors
df["cluster"] = df["product_id"].astype("category").cat.codes

df.groupby("price")["five_star"].agg(["size", "mean"]).loc[PRICE_ORDER].round(3)

### 3. Model specification

Main model (pre-specified in the proposal):

`five_star ~ price segment × skin type + category + highlight indicators`

Reference levels: Budget price, Normal skin, Moisturizers. Standard errors are **clustered by product**, because reviews of the same product share unobserved product quality and are not independent. Ignoring this would make the confidence intervals far too narrow.

The reduced model is identical but drops the interaction.

In [ ]:
REF = ("C(price, Treatment('Budget'))", "C(skin, Treatment('Normal'))", "C(cat, Treatment('Moisturizers'))")
hl_terms = " + ".join(HL)
F_MAIN = f"five_star ~ {REF[0]} * {REF[1]} + {REF[2]} + {hl_terms}"
F_REDUCED = f"five_star ~ {REF[0]} + {REF[1]} + {REF[2]} + {hl_terms}"

def fit_logit(formula, data):
    result = smf.logit(formula, data=data, missing="raise").fit(
        disp=False, maxiter=100, cov_type="cluster",
        cov_kwds={"groups": data["cluster"].to_numpy()})
    if not result.mle_retvals["converged"]:
        raise RuntimeError("Logistic model did not converge")
    return result

res_main = fit_logit(F_MAIN, df)
res_reduced = fit_logit(F_REDUCED, df)
print("Main model converged:", res_main.mle_retvals["converged"], "| parameters:", len(res_main.params))
print("Reduced model converged:", res_reduced.mle_retvals["converged"], "| parameters:", len(res_reduced.params))
print("Clusters (products):", df["cluster"].nunique())

### 4. Odds ratios with clustered 95% confidence intervals

In [ ]:
def tidy_label(term):
    t = term
    for long, short in [("C(price, Treatment('Budget'))[T.", "Price: "),
                        ("C(skin, Treatment('Normal'))[T.", "Skin: "),
                        ("C(cat, Treatment('Moisturizers'))[T.", "Category: ")]:
        t = t.replace(long, short)
    t = t.replace("]", "").replace(":Skin: ", " × Skin: ")
    t = t.replace("hl_", "Highlight: ").replace("highlights_missing", "No highlights listed")
    return t

def or_table(res):
    ci = res.conf_int()
    out = pd.DataFrame({
        "term": [tidy_label(t) for t in res.params.index],
        "model_term": res.params.index,
        "coef_log_odds": res.params.values,
        "se_clustered": res.bse.values,
        "z": res.tvalues.values,
        "odds_ratio": np.exp(res.params.values),
        "ci_low": np.exp(ci[0].values),
        "ci_high": np.exp(ci[1].values),
        "p_value": res.pvalues.values,
    })
    return out[out["term"] != "Intercept"].reset_index(drop=True)

or_main = or_table(res_main)
or_reduced = or_table(res_reduced)
or_main.to_csv(OUT_DIR / "odds_ratios_main_model.csv", index=False)
or_reduced.to_csv(OUT_DIR / "odds_ratios_reduced_model.csv", index=False)
or_main.drop(columns="model_term").round(3)

**Reading an odds ratio.** An OR of 1.10 means the odds of a five-star review are 10% higher than for the reference level, holding the other variables constant. An interval that crosses 1 is consistent with no association.

In the main model, the price odds ratios describe reviewers with **Normal** skin (the reference), and the skin-type odds ratios describe **Budget** products. The interaction terms show how much the price association shifts for each other skin type. Section 5 turns this into predicted probabilities, which are easier to present.

### 5. Does the price association differ by skin type?

In [ ]:
# Joint Wald test of all 12 interaction terms, using the clustered covariance
names = list(res_main.params.index)
inter_idx = [i for i, n in enumerate(names) if ":" in n]
R = np.zeros((len(inter_idx), len(names)))
for row, i in enumerate(inter_idx):
    R[row, i] = 1
wald = res_main.wald_test(R, scalar=True)
print(f"Interaction terms tested: {len(inter_idx)}")
WALD_CHI2, WALD_DF, WALD_P = float(wald.statistic), len(inter_idx), float(wald.pvalue)
print(f"Joint Wald test: chi2({WALD_DF}) = {WALD_CHI2:.2f}, p = {WALD_P:.4f}")
n_sig_inter = int((or_main.loc[or_main["term"].str.contains("×"), "p_value"] < 0.05).sum())
print(f"Individual interaction terms with p < 0.05: {n_sig_inter} of {WALD_DF} "
      f"(about {0.05 * WALD_DF:.1f} expected by chance alone if there were no interaction)")
pd.DataFrame([{"wald_chi2": WALD_CHI2, "wald_df": WALD_DF, "wald_p": WALD_P}]).to_csv(OUT_DIR / "wald_interaction.csv", index=False)


**Interpretation.** The joint Wald test asks whether all 12 skin type × price segment terms are zero at once. It does not reject this (p ≈ 0.50), so there is **insufficient evidence that the association between price segment and five-star ratings differs by skin type**. Individual interaction terms are not interpreted one by one: with 12 tests, about 0.6 would fall below p = 0.05 by chance even if no interaction existed. The predicted-probability figure below is descriptive and should be read in light of this test.

In [ ]:
# Average predicted probability for each skin type x price combination.
# Uncertainty: Monte Carlo draws of the coefficients from their clustered
# sampling distribution (the simulation idea from Week 8).
rng = np.random.default_rng(SEED)
base = df.sample(n=50_000, random_state=SEED)  # representative mix of the other covariates
from patsy import build_design_matrices
# statsmodels 0.14 stores a Patsy DesignInfo; 0.15 can use Formulaic.
X = getattr(res_main.model.data, "model_spec", None)
if X is None:
    X = res_main.model.data.design_info
draws = rng.multivariate_normal(res_main.params.values, res_main.cov_params().values, size=500)

rows = []
for skin in SKIN_ORDER:
    for price in PRICE_ORDER:
        scen = base.assign(skin=skin, price=price)
        Xs = np.asarray(build_design_matrices([X], scen, return_type="dataframe")[0])
        point = (1 / (1 + np.exp(-Xs @ res_main.params.values))).mean()
        sims = (1 / (1 + np.exp(-Xs @ draws.T))).mean(axis=0)
        rows.append({"skin": skin, "price": price, "pred": point,
                     "low": np.percentile(sims, 2.5), "high": np.percentile(sims, 97.5)})

pred = pd.DataFrame(rows)
pred.to_csv(OUT_DIR / "predicted_probabilities_skin_x_price.csv", index=False)
pred.pivot(index="skin", columns="price", values="pred").loc[SKIN_ORDER, PRICE_ORDER].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
offsets = np.linspace(-0.24, 0.24, len(SKIN_ORDER))
for k, skin in enumerate(SKIN_ORDER):
    s = pred[pred["skin"] == skin].set_index("price").loc[PRICE_ORDER]
    x = np.arange(len(PRICE_ORDER)) + offsets[k]
    ax.errorbar(x, s["pred"], yerr=[s["pred"] - s["low"], s["high"] - s["pred"]],
                fmt="o", ms=6, lw=1.6, capsize=0, color=PALETTE[k], label=skin)
ax.set_xticks(range(len(PRICE_ORDER)), ["Budget\n<$25", "Mid\n$25–49", "Premium\n$50–99", "Luxury\n$100+"])
ax.set_xlabel("Product price segment")
ax.set_ylabel("Predicted probability of a five-star review")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="x", visible=False)
ax.legend(title="Reviewer skin type", frameon=False, ncol=5, loc="upper center",
          bbox_to_anchor=(0.5, -0.30), fontsize=9, title_fontsize=9)
ax.set_title("Predicted five-star probability by price segment and skin type", loc="left", fontsize=11, color=INK, pad=32)
ax.text(0, 1.025, f"Average predictions from the main model; bars are 95% intervals (500 clustered coefficient draws).\n"
        f"Joint interaction test: χ²({WALD_DF}) = {WALD_CHI2:.2f}, p = {WALD_P:.2f}",
        transform=ax.transAxes, fontsize=8, color=INK2)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_predicted_skin_x_price.png", bbox_inches="tight")
plt.show()

### 6. Forest plot: category and product highlights

Category and highlight effects do not interact with anything, so their odds ratios read directly. Price and skin type are shown in the predicted-probability figure above instead, because with an interaction their raw odds ratios only describe one reference group.

In [ ]:
LABELS = {"Highlight: fragrance_free": "Fragrance free", "Highlight: clean": "Clean at Sephora",
          "Highlight: vegan": "Vegan", "Highlight: cruelty_free": "Cruelty free",
          "Highlight: hydrating": "Hydrating", "Highlight: dullness": "Good for: dullness",
          "Highlight: dryness": "Good for: dryness", "Highlight: anti_aging": "Good for: anti-aging",
          "No highlights listed": "No highlights listed"}
cat_df = or_main[or_main["term"].str.startswith("Category:")].copy()
cat_df["label"] = cat_df["term"].str.replace("Category: ", "", regex=False)
hl_df = or_main[or_main["term"].isin(LABELS)].copy()
hl_df["label"] = hl_df["term"].map(LABELS)
cat_df, hl_df = cat_df.sort_values("odds_ratio"), hl_df.sort_values("odds_ratio")

# Rows bottom to top: categories, a header, a gap, highlights, a header
blocks = [("Category (vs Moisturizers)", cat_df), ("Product highlights", hl_df)]
ypos, ylab, heads, plot_rows, y = [], [], [], [], 0
for head, block in blocks:
    for _, r in block.iterrows():
        plot_rows.append((y, r)); ypos.append(y); ylab.append(r["label"]); y += 1
    heads.append((y, head)); ypos.append(y); ylab.append(head); y += 1.6

fig, ax = plt.subplots(figsize=(7.5, 6.6))
for yy, r in plot_rows:
    sig = r["ci_low"] > 1 or r["ci_high"] < 1
    ax.hlines(yy, r["ci_low"], r["ci_high"], color=PALETTE[0], lw=1.8)
    ax.scatter(r["odds_ratio"], yy, s=36, color=PALETTE[0] if sig else "#fcfcfb",
               edgecolor=PALETTE[0], linewidth=1.6, zorder=3)
ax.axvline(1, color=INK2, lw=1, ls="--")
ax.set_xscale("log")
ax.set_xticks([0.6, 0.8, 1, 1.25, 1.5], ["0.6", "0.8", "1", "1.25", "1.5"])
ax.minorticks_off()
ax.set_yticks(ypos, ylab)
for tick, lab in zip(ax.get_yticklabels(), ylab):
    if lab in dict((h, 1) for _, h in heads):
        tick.set_fontweight("bold"); tick.set_color(INK)
ax.grid(axis="y", visible=False)
ax.set_ylim(-0.8, y - 1.2)
ax.set_xlabel("Odds ratio for a five-star review (log scale; 95% CI clustered by product)")
ax.set_title("Category and product claims associated with five-star reviews", loc="left", fontsize=11, color=INK, pad=22)
ax.text(0, 1.025, "Filled dots: interval excludes 1. Hollow dots: consistent with no association.",
        transform=ax.transAxes, fontsize=8, color=INK2)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_forest_category_highlights.png", bbox_inches="tight")
plt.show()

### 7. Evaluation on held-out products

- **Split:** 80% of products for training, 20% for testing (`GroupShuffleSplit` on `product_id`), so no product appears in both sets.
- **Models:** majority-class baseline, reduced model, main model. Both logistic models are refitted on the training products only.
- **Metrics:** ROC-AUC (threshold-free ranking quality) is the headline metric. Balanced accuracy, macro F1 and accuracy depend on the classification threshold, so they are reported at two thresholds:
    - **0.5**, the conventional default;
    - **the training five-star rate (about 0.64)**, which classifies a review as five-star only when the model rates it above the average review.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(df, groups=df["product_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
assert set(train["product_id"]).isdisjoint(set(test["product_id"]))
print(f"Train: {len(train):,} reviews, {train['product_id'].nunique()} products | "
      f"Test: {len(test):,} reviews, {test['product_id'].nunique()} products")
print(f"Five-star rate  train {train['five_star'].mean():.3f} | test {test['five_star'].mean():.3f}")

# Refit on training products only
m_main = fit_logit(F_MAIN, train)
m_red = fit_logit(F_REDUCED, train)
THRESHOLDS = {"0.5 (default)": 0.5, f"{train['five_star'].mean():.3f} (training five-star rate)": train["five_star"].mean()}
y_test = test["five_star"].to_numpy()

p_red = m_red.predict(test).to_numpy()
p_main = m_main.predict(test).to_numpy()

def threshold_metrics(name, thr_name, prob, thr):
    lbl = (prob >= thr).astype(int)
    return {"model": name, "threshold": thr_name,
            "ROC-AUC": roc_auc_score(y_test, prob),
            "Balanced accuracy": balanced_accuracy_score(y_test, lbl),
            "Macro F1": f1_score(y_test, lbl, average="macro"),
            "Accuracy": accuracy_score(y_test, lbl),
            "Share predicted five-star": lbl.mean()}

ones = np.ones_like(y_test)
rows = [{"model": "Baseline (always five-star)", "threshold": "n/a", "ROC-AUC": 0.5,
         "Balanced accuracy": balanced_accuracy_score(y_test, ones),
         "Macro F1": f1_score(y_test, ones, average="macro"),
         "Accuracy": accuracy_score(y_test, ones), "Share predicted five-star": 1.0}]
for name, prob in [("Reduced model", p_red), ("Main model", p_main)]:
    for thr_name, thr in THRESHOLDS.items():
        rows.append(threshold_metrics(name, thr_name, prob, thr))
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT_DIR / "evaluation_metrics.csv", index=False)
print(f"Test-set predicted probabilities (main model): min {p_main.min():.3f}, median {np.median(p_main):.3f}, max {p_main.max():.3f}")
metrics.round(3)

**Why the two thresholds differ.** The model's predicted probabilities sit in a narrow band around the base rate (see the printed range), because product attributes shift the odds only slightly. At **0.5**, every (or almost every) predicted probability is above the cut-off, so the model labels the test reviews five-star and reproduces the baseline: accuracy equals the five-star share, and balanced accuracy is 0.5. At the **training five-star rate**, the model splits reviews into above- and below-average, which improves macro F1 relative to the baseline but lowers raw accuracy, because many five-star reviews fall just below the cut-off.

Neither threshold changes ROC-AUC, which measures ranking quality. **ROC-AUC is about 0.52 against 0.50 for the baseline: the model ranks five-star reviews only slightly better than chance.**

In [ ]:
# Confusion matrices for the main model at both thresholds
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
cms = {}
for ax, (thr_name, thr) in zip(axes, THRESHOLDS.items()):
    cm = confusion_matrix(y_test, (p_main >= thr).astype(int))
    cms[thr_name] = cm
    row_share = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(row_share, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}\n({row_share[i, j]:.0%} of row)", ha="center", va="center",
                    fontsize=9, color="white" if row_share[i, j] > 0.6 else INK)
    ax.set_xticks([0, 1], ["Not five-star", "Five-star"])
    ax.set_yticks([0, 1], ["Not five-star", "Five-star"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.grid(False)
    ax.set_title(f"Threshold {thr_name}", fontsize=9.5, color=INK, loc="left")
fig.suptitle("Main model confusion matrices on held-out products", x=0.02, ha="left", fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_confusion_matrix_main.png", bbox_inches="tight")
plt.show()

for thr_name, cm in cms.items():
    tag = "default_0.5" if thr_name.startswith("0.5") else "training_rate"
    pd.DataFrame(cm, index=["Actual: not five-star", "Actual: five-star"],
                 columns=["Predicted: not five-star", "Predicted: five-star"]).to_csv(OUT_DIR / f"confusion_matrix_main_{tag}.csv")

### 8. Summary: associations versus prediction

These are two different findings and should not be mixed up:

- **Statistical associations (Sections 4–6).** Several product characteristics are associated with five-star ratings, with clustered confidence intervals that exclude 1. These describe patterns in the data; they are not causal effects, because the data are observational.
- **Predictive performance (Section 7).** The same variables predict individual five-star reviews only slightly better than chance on held-out products. A variable can be clearly associated with the outcome on average while explaining little of the variation between individual reviews.

Q&A notes:

- **Why logistic regression?** The outcome is binary and the aim is interpretable associations; odds ratios with confidence intervals answer the research question directly.
- **Why cluster by product?** Reviews of the same product are correlated. Treating about 1.09 million reviews as independent would make the intervals far too narrow.
- **Why split by product?** A random split would put reviews of the same product in both sets, letting the model memorise product quality and inflating test scores.
- **Why is predictive power weak?** Review-time information (text, recommendation, helpfulness) is excluded to avoid leakage, and the remaining product attributes explain little of individual satisfaction.

The cell below writes `Part_E_Report_Notes.md` and `Part_E_Handoff_to_F.md`, filling every number directly from the objects computed above so the documents always match the results.

In [ ]:
def fmt_or(r):
    p = "< 0.001" if r["p_value"] < 0.001 else f"= {r['p_value']:.3f}"
    return f"{r['odds_ratio']:.2f} (95% CI {r['ci_low']:.2f}–{r['ci_high']:.2f}, p {p})"

main_terms = or_main[~or_main["term"].str.contains("×")]
sig_main = main_terms[(main_terms["ci_low"] > 1) | (main_terms["ci_high"] < 1)].sort_values("p_value")
m = metrics.set_index(["model", "threshold"])
thr_keys = list(THRESHOLDS)
base_row = metrics.iloc[0]
auc_main = m.loc[("Main model", thr_keys[0]), "ROC-AUC"]
auc_red = m.loc[("Reduced model", thr_keys[0]), "ROC-AUC"]
pp = pred.pivot(index="skin", columns="price", values="pred").loc[SKIN_ORDER, PRICE_ORDER]

metric_lines = ["| Model | Threshold | ROC-AUC | Balanced accuracy | Macro F1 | Accuracy | Share predicted five-star |",
                "| --- | --- | --- | --- | --- | --- | --- |"]
for _, r in metrics.iterrows():
    metric_lines.append(f"| {r['model']} | {r['threshold']} | {r['ROC-AUC']:.3f} | {r['Balanced accuracy']:.3f} | "
                        f"{r['Macro F1']:.3f} | {r['Accuracy']:.3f} | {r['Share predicted five-star']:.1%} |")
metric_table = "\n".join(metric_lines)

sig_table = "\n".join(["| Term | Odds ratio |", "| --- | --- |"] +
                      [f"| {r['term']} | {fmt_or(r)} |" for _, r in sig_main.iterrows()])
pp_table = "\n".join(["| Skin type | " + " | ".join(PRICE_ORDER) + " |",
                      "| --- |" + " --- |" * len(PRICE_ORDER)] +
                     [f"| {sk} | " + " | ".join(f"{pp.loc[sk, pr]:.1%}" for pr in PRICE_ORDER) + " |"
                      for sk in SKIN_ORDER])

share_05 = m.loc[("Main model", thr_keys[0]), "Share predicted five-star"]
f1_rate = m.loc[("Main model", thr_keys[1]), "Macro F1"]
acc_rate = m.loc[("Main model", thr_keys[1]), "Accuracy"]

notes = "\n".join([
    "# Part E report notes: main model and evaluation",
    "",
    "All numbers below are generated by `03_main_model_E.ipynb`; rerun the notebook to refresh them.",
    "",
    "## Data and model",
    "",
    f"- Reviews: {len(df):,}; products (clusters): {df['product_id'].nunique():,}; five-star rate: {df['five_star'].mean():.1%}.",
    "- Main model: logistic regression of `five_star` on price segment × skin type, category and nine highlight indicators. "
    "References: Budget price, Normal skin, Moisturizers.",
    "- Standard errors are clustered by product. Reduced model: identical without the interaction.",
    "- Full coefficients, clustered standard errors, odds ratios, 95% CIs and p-values: `odds_ratios_main_model.csv`.",
    "",
    "## Statistical associations",
    "",
    "Main-effect terms whose 95% confidence interval excludes 1:",
    "",
    sig_table,
    "",
    "Category and highlight odds ratios hold the other variables constant. Because of the interaction, the price odds ratios "
    "describe Normal-skin reviewers and the skin-type odds ratios describe Budget products; use the predicted probabilities "
    "below to discuss price and skin type.",
    "",
    "These are associations in observational data, not causal effects.",
    "",
    "## Skin type × price segment interaction",
    "",
    f"- Joint Wald test of the {WALD_DF} interaction terms: χ²({WALD_DF}) = {WALD_CHI2:.2f}, p = {WALD_P:.4f}.",
    "- There is insufficient evidence that the association between price segment and five-star ratings differs by skin type.",
    f"- {n_sig_inter} of {WALD_DF} individual interaction terms {'has' if n_sig_inter == 1 else 'have'} p < 0.05, about what chance alone would produce "
    f"({0.05 * WALD_DF:.1f} expected). They are not interpreted individually.",
    "",
    "Average predicted probability of a five-star review (main model; 95% intervals in `predicted_probabilities_skin_x_price.csv`):",
    "",
    pp_table,
    "",
    "## Predictive performance on held-out products",
    "",
    f"Split: 80/20 by product (seed {SEED}); {len(train):,} training and {len(test):,} test reviews; no product appears in both sets.",
    "",
    metric_table,
    "",
    f"- ROC-AUC is {auc_main:.3f} for the main model and {auc_red:.3f} for the reduced model, against 0.500 for the baseline: "
    "the model ranks five-star reviews only slightly better than chance.",
    (f"- At the 0.5 threshold every predicted probability is above 0.5, so the model labels all test reviews five-star and its threshold-based metrics are identical to the baseline's."
     if share_05 >= 0.9995 else
     f"- At the 0.5 threshold the model labels {share_05:.1%} of test reviews five-star, so it behaves almost like the baseline."),
    f"- At the training five-star rate ({THRESHOLDS[thr_keys[1]]:.3f}), macro F1 rises to {f1_rate:.3f} "
    f"(baseline {base_row['Macro F1']:.3f}) but accuracy falls to {acc_rate:.3f}, because many five-star reviews fall just below the cut-off.",
    "",
    "## How to phrase the conclusion",
    "",
    "- Product category and some product claims are associated with the likelihood of a five-star review, but these associations are modest.",
    "- The same variables have weak out-of-sample predictive power: product attributes explain little of whether an individual reviewer gives five stars.",
    "- There is no evidence that the price–rating association differs by skin type.",
    "- Avoid causal wording such as \"Clean at Sephora products cause lower ratings\"; use \"are associated with\".",
    "",
    "## Figures",
    "",
    "- `fig_forest_category_highlights.png`: odds ratios for category and highlights.",
    "- `fig_predicted_skin_x_price.png`: predicted probabilities by price segment and skin type.",
    "- `fig_confusion_matrix_main.png`: main-model confusion matrices at both thresholds.",
    "",
])

rec_missing = df["recommended"].isna().mean()
rec_n = int(df["recommended"].notna().sum())
rec_rate = df["recommended"].mean()

handoff = "\n".join([
    "# Part E to Part F handoff",
    "",
    "Part F reruns the Part E pipeline with (1) the recommendation flag as the outcome and (2) log price instead of price segment. "
    "Keep everything else identical so the results are directly comparable.",
    "",
    "## What to reuse from `03_main_model_E.ipynb`",
    "",
    "Run Sections 1–3 unchanged (setup, loading, the `price`, `skin`, `cat`, `HL` and `cluster` columns, `REF`, `hl_terms`, `F_MAIN`). Then reuse:",
    "",
    "- `fit_logit(formula, data)`: logistic regression with product-clustered standard errors.",
    "- `or_table(res)`: coefficients, clustered SEs, odds ratios, 95% CIs and p-values.",
    "- The joint Wald test code in Section 5.",
    f"- The evaluation code in Section 7: the same `GroupShuffleSplit`, `SEED = {SEED}`, and the two thresholds (0.5 and the training outcome rate).",
    "",
    "## 1. Secondary outcome: recommendation",
    "",
    "```python",
    "rec = df[df[\"recommended\"].notna()].copy()",
    "rec[\"recommended\"] = rec[\"recommended\"].astype(int)",
    "rec[\"cluster\"] = rec[\"product_id\"].astype(\"category\").cat.codes  # recompute after filtering",
    "",
    "F_REC = F_MAIN.replace(\"five_star ~\", \"recommended ~\")",
    "res_rec = fit_logit(F_REC, rec)",
    "or_rec = or_table(res_rec)",
    "```",
    "",
    f"- `recommended` is missing for {rec_missing:.1%} of reviews, leaving {rec_n:,} reviews. State this in the report.",
    f"- The recommendation rate is {rec_rate:.1%}, so the second threshold is the training recommendation rate, alongside 0.5.",
    "- For a like-for-like comparison, also refit `F_MAIN` on the same `rec` rows, so differences are not caused by the smaller sample.",
    "- Recommendation and five-star rating measure different things. Report differences between them as findings, not as failures of robustness.",
    "",
    "## 2. Sensitivity: log price instead of price segment",
    "",
    "```python",
    "F_LOGP = f\"five_star ~ log_price * {REF[1]} + {REF[2]} + {hl_terms}\"",
    "res_logp = fit_logit(F_LOGP, df)",
    "or_logp = or_table(res_logp)",
    "```",
    "",
    f"- Keep the interaction with skin type so the specification mirrors the main model. It now has {len(SKIN_ORDER) - 1} terms, "
    f"so its joint Wald test has {len(SKIN_ORDER) - 1} degrees of freedom.",
    "- Do not include `price_segment` and `log_price` in the same model.",
    "- The `log_price` odds ratio is per one-unit increase in log price (about a 2.7-fold price increase). "
    "Report `exp(coef × log(2))` for the change per doubling of price.",
    "",
    "## Part E reference numbers to compare against",
    "",
    f"- Five-star main model, joint interaction test: χ²({WALD_DF}) = {WALD_CHI2:.2f}, p = {WALD_P:.4f}.",
    f"- Main model held-out ROC-AUC: {auc_main:.3f}.",
    "",
    "## Do not add",
    "",
    "- New predictors, other model types or a different split. Part F changes only the outcome or the price variable.",
    "",
])

(OUT_DIR / "Part_E_Report_Notes.md").write_text(notes, encoding="utf-8")
(OUT_DIR / "Part_E_Handoff_to_F.md").write_text(handoff, encoding="utf-8")
print(notes)